In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.getcwd().endswith("/quotaclimat"):
    os.chdir("../../../../..")

repo_root_path = os.path.abspath(os.path.dirname(os.getcwd()))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path

In [ ]:
import time
from datetime import datetime, timedelta, timezone

import boto3
from botocore.config import Config as BotoConfig

from quotaclimat.data_ingestion.advertising.s02_exportation.ad_bucket import (
    AD_S3_PREFIX,
)
from quotaclimat.data_ingestion.advertising.tools.interactive_tqdm import (
    interactive_tqdm,
)

`fs.find()` (s3fs) walks the bucket folder by folder -- one S3 API call per `ads/<ad_id>/` folder -- which is fine for a handful of folders but effectively hangs once there are tens of thousands of them. A plain, flat `list_objects_v2` paginator instead lists ~1000 keys per call regardless of how many folders they're spread across, so it's used directly here (via `boto3`, same credentials/endpoint as the rest of the pipeline).

In [ ]:
REGION = "fr-par"
ENDPOINT_URL = f"https://s3.{REGION}.scw.cloud"

s3_client = boto3.client(
    "s3",
    region_name=REGION,
    aws_access_key_id=os.environ["BUCKET"],
    aws_secret_access_key=os.environ["BUCKET_SECRET"],
    endpoint_url=ENDPOINT_URL,
    config=BotoConfig(signature_version="s3v4", max_pool_connections=50),
)

since = datetime.now(timezone.utc) - timedelta(days=1)
BUCKET_NAME = 'advertising-detection' # Warning it's the prod
BUCKET_NAME, AD_S3_PREFIX, since

In [ ]:
def list_new_files_since(
    s3_client, bucket_name: str, prefix: str, since: datetime
) -> list[dict]:
    """List objects under `prefix` whose LastModified is at or after `since`, newest first.

    Scans the whole prefix (keys aren't sorted by date), but as one flat, paginated
    listing rather than a per-folder walk -- see the note above. Progress (objects
    scanned so far, and the scan rate) is shown live so you can tell whether this is a
    3-minute or 3-day job before waiting it out.
    """
    paginator = s3_client.get_paginator("list_objects_v2")
    new_files = []

    start_time = time.monotonic()
    # interactive_tqdm falls back to logger.info() lines whenever stdout isn't a tty,
    # which is always true in a notebook -- disable=False forces the real bar instead.
    with interactive_tqdm(desc="Scanning bucket", unit=" obj", disable=False) as progress:
        for page in paginator.paginate(
            Bucket=bucket_name,
            Prefix=f"{prefix}/",
            PaginationConfig={"PageSize": 1000},
        ):
            contents = page.get("Contents", [])
            progress.update(len(contents))

            for obj in contents:
                if obj["LastModified"] >= since:
                    new_files.append(obj)
                    progress.count("matched")

    elapsed = time.monotonic() - start_time
    print(f"Scanned {progress.n:,} objects in {elapsed:.0f}s ({progress.n / elapsed:.0f} obj/s)")

    new_files.sort(key=lambda obj: obj["LastModified"], reverse=True)
    return new_files

In [ ]:
new_files = list_new_files_since(s3_client, BUCKET_NAME, AD_S3_PREFIX, since)
print(f"{len(new_files)} new file(s) since {since.isoformat()}")

for obj in new_files:
    print(obj["LastModified"], obj["Key"], obj["Size"])